##### What is data loading?

Data loading means transferring data from a source system into a target system.

##### Full Load

Every time the pipeline runs, it loads the complete source data.

**Problem:** Every run processes old records again, which can increase processing time and cost.

##### Incremental Load
Only new or modified records are porcessed.

* Incremental loading reduces unnecessory data processing.


##### Full Load vs Incremental Load

| Aspect | Full Load | Incremental Load |
|---|---|---|
| **What is loaded** | Entire source dataset, every run | Only new/changed records since last run |
| **Processing cost** | High — old data reprocessed repeatedly | Low — only the delta is processed |
| **Runtime** | Grows with total table size | Stays roughly constant per run |
| **Complexity** | Simple to build (no state needed) | Needs a way to track "what's already loaded" |
| **Best for** | Small tables, one-time backfills, full refresh | Large, continuously growing tables |

##### Watermark

A **watermark** is a saved marker (usually a timestamp or monotonically increasing ID) that records *how far you have already loaded*. Each incremental run:

1. Reads the watermark (last successfully loaded value).
2. Loads only rows where the marker column is **greater than** the watermark (e.g. `updated_at > '2024-05-01 12:00:00'` or `id > 1024`).
3. Updates the watermark to the maximum value seen in this run, so the next run starts from there.

```
Source table                     Watermark = 100
┌────┬───────────────┐            ┌──────────────────┐
| id | updated_at     │            | Run 1: id > 100   │──> loads 101–150
├────┼───────────────┤   ──Run──>  | watermark → 150   │
| 98 | 2024-04-30 ... │  (skipped) └──────────────────┘
| 101| 2024-05-01 ... │            ┌──────────────────┐
| 150| 2024-05-02 ... │            | Run 2: id > 150   │──> loads 151–200
└────┴───────────────┘            | watermark → 200   │
                                 └──────────────────┘
```

**Key points:**

* The watermark column must be **monotonically increasing** (a timestamp of last update, an auto-increment ID, or a change sequence number).
* The watermark must be stored **durably** (a small state table, Delta table metadata, or the checkpoint of a streaming pipeline) and updated **only after** a successful load — otherwise rows can be lost.
* On Databricks, streaming reads (Auto Loader, `readChangeFeed`) track watermarks for you via checkpoints; in batch pipelines you manage the watermark table yourself.
* Watch out for **late-arriving or updated rows**: a plain `id > watermark` filter misses updates to old rows, so a watermark on `updated_at` (with the column maintained on updates) is often safer — this is exactly what Delta **Change Data Feed** solves.


In [0]:
# Full Load vs Incremental Load (with watermark) — example with 5 records
# Source table has 5 rows total: 3 arrive on day 1, 2 more arrive on day 2.

from pyspark.sql.functions import col

# --- The source table (5 records) ---
batch_day1 = [(1, "Alice"), (2, "Bob"), (3, "Carol")]   # arrives on day 1
batch_day2 = [(4, "Dave"), (5, "Eve")]                    # arrives on day 2

source_day1 = spark.createDataFrame(batch_day1, "id INT, name STRING")
source_day2 = spark.createDataFrame(batch_day1 + batch_day2, "id INT, name STRING")

print("Source on day 1 (3 rows):")
source_day1.show()
print("Source on day 2 (5 rows — 2 new since day 1):")
source_day2.show()

# =====================================================================
# APPROACH 1: FULL LOAD — every run reads the ENTIRE source again
# =====================================================================
print("=" * 60)
print("FULL LOAD")
print("=" * 60)

full_run1 = source_day1                       # run 1: processes 3 rows
full_run2 = source_day2                       # run 2: reprocesses ALL 5 rows (3 old + 2 new)

print(f"Run 1: loaded {full_run1.count()} rows  (Alice, Bob, Carol)")
print(f"Run 2: loaded {full_run2.count()} rows  (re-processing 3 OLD rows + 2 new rows)")
print(f"Total rows processed over 2 runs: {full_run1.count() + full_run2.count()}")

# =====================================================================
# APPROACH 2: INCREMENTAL LOAD with WATERMARK on `id`
# Rule: load only rows where id > watermark, then move watermark to max(id)
# =====================================================================
print("=" * 60)
print("INCREMENTAL LOAD (watermark on id)")
print("=" * 60)

watermark = 0          # durable marker: nothing loaded yet
total_processed = 0

for run_no, source in [(1, source_day1), (2, source_day2)]:
    # 1. Load only the NEW rows: id > watermark
    new_rows = source.filter(col("id") > watermark)
    n = new_rows.count()
    total_processed += n

    # 2. Advance the watermark to the max id seen in THIS run
    #    (only after the load succeeds — otherwise rows can be lost)
    watermark = source.agg({"id": "max"}).collect()[0][0] or watermark

    print(f"Run {run_no}: watermark={watermark - n} -> loaded {n} new row(s) "
          f"-> new watermark = {watermark}")
    new_rows.show()

print(f"Total rows processed over 2 runs: {total_processed}")
print()
print("Compare: full load processed 8 rows (3 + 5), incremental processed only 5.")
print("As the table grows, the gap grows — full load cost rises with table size,")
print("while incremental load stays roughly constant per run.")


In [0]:
# CDC (Change Data Capture) — detecting INSERT, UPDATE, DELETE via HASH COMPARISON
#
# Scenario: we get a FULL snapshot of the source each day (no change feed available).
# How do we figure out what changed without comparing every column one-by-one?
#
#   -> Compute one HASH per row (a compact "fingerprint" of all tracked columns).
#
#      key in source only        -> INSERT
#      key in target only        -> DELETE (row disappeared from source)
#      hash(source) != hash(target) -> UPDATE
#      hash matches              -> UNCHANGED (skip it)

from pyspark.sql import functions as F

# =====================================================================
# TARGET = what we loaded yesterday (current state, 3 rows)
# =====================================================================
target = spark.createDataFrame(
    [(1, "Alice", "alice@x.com", "Mumbai"),
     (2, "Bob",   "bob@x.com",   "Pune"),
     (3, "Carol", "carol@x.com", "Delhi")],
    "id INT, name STRING, email STRING, city STRING"
)

# =====================================================================
# SOURCE = today's snapshot (5 possible outcomes for our 3 keys + 1 new)
#   id=1: identical          -> UNCHANGED
#   id=2: email changed      -> UPDATE
#   id=3: missing from source -> DELETE
#   id=4: brand new          -> INSERT
# =====================================================================
source = spark.createDataFrame(
    [(1, "Alice", "alice@x.com", "Mumbai"),
     (2, "Bob",   "bob@new.com", "Pune"),
     (4, "Dave",  "dave@x.com",  "Chennai")],
    "id INT, name STRING, email STRING, city STRING"
)

print("TARGET (yesterday's state):")
target.show()
print("SOURCE (today's snapshot):")
source.show()

# =====================================================================
# Step 1: HASH each row — all tracked columns collapsed into one value
#   concat_ws + coalesce keeps NULLs from silently changing the hash,
#   and '||' separator stops "ab"+"c" colliding with "a"+"bc".
# =====================================================================
tracked_cols = ["name", "email", "city"]

def with_hash(df):
    return df.withColumn(
        "row_hash",
        F.sha2(
            F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit(""))
                                for c in tracked_cols]),
            256
        )
    )

target_h = with_hash(target)
source_h = with_hash(source)

print("Rows with their hashes:")
target_h.select("id", "name", "email", "row_hash").show(truncate=False)
source_h.select("id", "name", "email", "row_hash").show(truncate=False)

# =====================================================================
# Step 2: FULL OUTER JOIN on the key and classify every row
# =====================================================================
changes = (
    target_h.alias("t")
    .join(source_h.alias("s"), "id", "full_outer")
    .select(
        "id",
        F.when(F.col("t.row_hash").isNull(),              F.lit("INSERT"))   # key only in source
         .when(F.col("s.row_hash").isNull(),              F.lit("DELETE"))    # key only in target
         .when(F.col("t.row_hash") != F.col("s.row_hash"), F.lit("UPDATE"))  # fingerprint differs
         .otherwise(F.lit("UNCHANGED"))
        .alias("change_type")
    )
)

print("Detected changes:")
changes.show()
changes.groupBy("change_type").count().show()

# =====================================================================
# Step 3: APPLY the changes -> new target state
#   keep UNCHANGED target rows, take INSERT/UPDATE rows from source,
#   drop DELETE rows (they're not in the source anyway)
# =====================================================================
keep_ids = changes.filter("change_type = 'UNCHANGED'").select("id")
new_rows = source_h.join(changes.filter("change_type IN ('INSERT','UPDATE')"), "id")

new_target = (
    target.join(keep_ids, "id").select("id", "name", "email", "city")
    .unionByName(new_rows.select("id", "name", "email", "city"))
)

print("NEW TARGET state after applying INSERT/UPDATE/DELETE:")
new_target.orderBy("id").show()

print("Why hash comparison? With 3 columns you could compare each column, but real tables")
print("have 50+ columns. sha2 collapses the whole row into ONE value, so change detection")
print("stays a single 't.hash != s.hash' check no matter how wide the table is — and it")
print("updates the watermark only where it actually changed, so unchanged rows are skipped.")
print("Caveat: hashing compares SNAPSHOTS (full-scan each run). If your source supports a")
print("change feed (e.g. Delta Change Data Feed), prefer it — it emits only the INSERT /")
print("UPDATE / DELETE events directly, with no comparison needed.")

In [0]:
# Delta Lake MERGE — the incremental UPSERT pattern
#
# MERGE = "upsert": a row is INSERTed if its key doesn't exist in the target yet,
# and UPDATEd in place if it does. This is the standard way to apply an
# incremental (delta) batch to a target table without reprocessing history.
#
# Why Delta MERGE instead of full overwrite?
#   * Only the files containing matched rows get rewritten (cheap, fast).
#   * It's atomic — readers never see a half-applied batch.
#   * Works with the watermark idea from earlier: source = rows WHERE id > watermark.

from pyspark.sql.functions import col

# =====================================================================
# TARGET: current state, stored as a Delta table (3 rows loaded earlier)
# =====================================================================
target_df = spark.createDataFrame(
    [(1, "Alice", "Mumbai"),
     (2, "Bob",   "Pune"),
     (3, "Carol", "Delhi")],
    "id INT, name STRING, city STRING"
)

target_df.write.format("delta").mode("overwrite") \
    .saveAsTable("demo_customers_target")

print("TARGET before merge:")
display(spark.table("demo_customers_target").orderBy("id"))

# =====================================================================
# INCREMENTAL batch (e.g. rows past the watermark, or detected changes):
#   id=2 -> exists in target, city changed  -> UPDATE
#   id=4 -> brand new key                    -> INSERT
#   id=1, id=3 -> not in the batch at all    -> untouched (no rewrite)
# =====================================================================
increment_df = spark.createDataFrame(
    [(2, "Bob",   "Chennai"),   # update: Bob moved
     (4, "Dave", "Hyderabad")],  # insert: new customer
    "id INT, name STRING, city STRING"
)

increment_df.createOrReplaceTempView("increment_batch")

# =====================================================================
# MERGE: upsert the incremental rows into the target
# =====================================================================
spark.sql("""
    MERGE INTO demo_customers_target AS t
    USING increment_batch AS s
       ON t.id = s.id
    WHEN MATCHED AND (t.city <> s.city OR t.name <> s.name)
        THEN UPDATE SET *
    WHEN NOT MATCHED
        THEN INSERT *
""")

print("TARGET after merge (Bob updated in place, Dave inserted, Alice/Carol untouched):")
display(spark.table("demo_customers_target").orderBy("id"))

# =====================================================================
# MERGE can also DELETE: keys present in target but missing from the
# source batch. Careful — only safe if the batch is a FULL key set.
# (Our batch is incremental, so we do NOT enable this clause here.)
# =====================================================================
print("""
Full pattern for snapshot-style batches:

    MERGE INTO target AS t
    USING source_batch AS s
       ON t.id = s.id
    WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET *
    WHEN MATCHED AND s.is_deleted = true     THEN DELETE
    WHEN NOT MATCHED AND s.is_deleted = false THEN INSERT *
    WHEN NOT MATCHED BY SOURCE AND <batch covers everything> THEN DELETE

Combine with a watermark: feed MERGE only rows WHERE updated_at > watermark,
then advance the watermark after the MERGE commits.
""")

In [0]:
# SCD Type 1 — keep only the CURRENT (latest) state, overwriting old values
#
# SCD Type 1 = "no history": when a tracked attribute changes, UPDATE the row
# in place. The target always reflects the newest snapshot of each key.
# (Compare with SCD Type 2, which appends a new version row + valid-from/to
# timestamps so full history is retained.)
#
# Implementation = Delta MERGE (upsert):
#   key not in target            -> INSERT
#   key in target + values differ -> UPDATE (old value is gone forever)
#   key in target + same values   -> UNCHANGED (skipped, files not rewritten)

# =====================================================================
# TARGET = current state (what SCD1 looks like today), 3 rows
# =====================================================================
target_df = spark.createDataFrame(
    [(1, "Alice", "Mumbai",  "alice@x.com"),
     (2, "Bob",   "Pune",    "bob@x.com"),
     (3, "Carol", "Delhi",   "carol@x.com")],
    "id INT, name STRING, city STRING, email STRING"
)

target_df.write.format("delta").mode("overwrite") \
    .saveAsTable("scd1_customers")

print("TARGET before (current state):")
display(spark.table("scd1_customers").orderBy("id"))

# =====================================================================
# SOURCE = incremental batch detected since the last run
#   id=1: unchanged         -> skipped
#   id=2: city changed      -> UPDATE in place (old city 'Pune' is LOST —
#                              that's what "Type 1, no history" means)
#   id=4: brand new key     -> INSERT
# =====================================================================
source_df = spark.createDataFrame(
    [(1, "Alice", "Mumbai",    "alice@x.com"),
     (2, "Bob",   "Chennai",   "bob@x.com"),
     (4, "Dave",  "Hyderabad", "dave@x.com")],
    "id INT, name STRING, city STRING, email STRING"
)

source_df.createOrReplaceTempView("scd1_source_batch")

# =====================================================================
# MERGE — the heart of SCD Type 1
#   * match on the business key (id)
#   * WHEN MATCHED AND <change condition> THEN UPDATE SET *  -> unchanged
#     rows are NOT rewritten (fewer files touched, faster merges)
#   * WHEN NOT MATCHED THEN INSERT *  -> brand-new keys
# =====================================================================
spark.sql("""
    MERGE INTO scd1_customers AS t
    USING scd1_source_batch AS s
       ON t.id = s.id
    WHEN MATCHED AND (t.name <> s.name OR t.city <> s.city
                      OR t.email <> s.email)
        THEN UPDATE SET *
    WHEN NOT MATCHED
        THEN INSERT *
""")

print("TARGET after MERGE (Bob updated in place, Dave inserted, Alice untouched):")
display(spark.table("scd1_customers").orderBy("id"))

# =====================================================================
# Handling DELETES in SCD Type 1
#   A plain incremental batch can't tell "key missing" from "key deleted".
#   Sources that publish change events add a tombstone/op flag:
# =====================================================================
print("""
DELETE handling — when the batch carries a change-type flag:

    MERGE INTO scd1_customers AS t
    USING change_batch AS s
       ON t.id = s.id
    WHEN MATCHED AND s.op = 'DELETE'       THEN DELETE
    WHEN MATCHED AND s.op = 'UPDATE'       THEN UPDATE SET *
    WHEN NOT MATCHED AND s.op <> 'DELETE'  THEN INSERT *

If instead you receive FULL snapshots, classify rows with the hash-comparison
full-outer join (see the CDC cell above), then MERGE only the INSERT/UPDATE/
DELETE rows — never reprocess the UNCHANGED majority.
""")

# =====================================================================
# Production notes:
#   * Advance the watermark only AFTER the MERGE commits successfully.
#   * For wide tables, pre-compute sha2 row hashes so the MATCHED condition
#     is a single t.row_hash <> s.row_hash check.
#   * Delta MERGE is atomic — readers never see a half-applied batch.
#   * Need full history instead of "latest state"? That's SCD Type 2
#     (append new version rows + valid_from/valid_to per key) — different
#     pattern, see the auto-CDC streaming table approach.
# =====================================================================


In [0]:
# SCD Type 2 — keep FULL HISTORY of every change (append new version rows)
#
# SCD Type 2 = "with history": when a tracked attribute changes, we do NOT overwrite.
# Instead we close the old version (is_current = false, valid_to = change date)
# and append a NEW row as the current version. Full history is queryable.
#
# SCD1 vs SCD2, same event (Bob moves from Pune to Chennai on 2024-06-01):
#   SCD1: the single Bob row is UPDATED in place -> old city 'Pune' is lost.
#   SCD2: the 'Pune' row is CLOSED (valid_to = 2024-06-01, is_current = false)
#         and a NEW 'Chennai' row is INSERTED (valid_from = 2024-06-01).

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =====================================================================
# TARGET = SCD2 dimension table (current + historical versions), 3 rows
#   sk         = surrogate key (unique per version row, never changes)
#   id         = business key (same across all versions of one entity)
#   valid_from / valid_to = when this version was true (NULL = still true)
#   is_current = true only for the newest version of each key
# =====================================================================
target_df = spark.createDataFrame(
    [(100, 1, "Alice", "Mumbai", "2024-01-01", None,      True),
     (101, 2, "Bob",   "Pune",   "2024-01-01", None,      True),
     (102, 3, "Carol", "Delhi",  "2024-01-01", None,      True)],
    "sk INT, id INT, name STRING, city STRING, "
    "valid_from STRING, valid_to STRING, is_current BOOLEAN"
)

target_df.write.format("delta").mode("overwrite") \
    .saveAsTable("scd2_customers")

print("TARGET before (all rows are current):")
display(spark.table("scd2_customers").orderBy("sk"))

# =====================================================================
# SOURCE = today's incremental batch (effective_date = when it became true)
#   id=1: unchanged   -> no action at all (skipped)
#   id=2: city changed -> close old version + insert new version
#   id=4: brand new    -> insert as current
# =====================================================================
effective_date = "2024-06-01"

source_df = spark.createDataFrame(
    [(1, "Alice", "Mumbai"),
     (2, "Bob",   "Chennai"),
     (4, "Dave",  "Hyderabad")],
    "id INT, name STRING, city STRING"
)

current = spark.table("scd2_customers").where("is_current = true")

# Detect which source rows actually CHANGED vs the current target version
changed = (
    source_df.alias("s")
    .join(current.alias("t"), "id", "left")
    .where("t.id IS NULL OR t.name <> s.name OR t.city <> s.city")
    .select("s.id", "s.name", "s.city")
)
print("Detected changed / new rows:")
display(changed.orderBy("id"))

# =====================================================================
# Build the MERGE source using the classic "split row" trick:
# Delta MERGE can't UPDATE and INSERT for the SAME matched key in one pass,
# so each changed key emits TWO rows:
#   merge_key =  id  (action='CLOSE')  -> matches the current row, closes it
#   merge_key = -id  (action='INSERT') -> never matches, inserts new version
# Unchanged rows are simply not in the merge source -> untouched (no rewrite)
# =====================================================================
closes = changed.select(
    F.col("id").alias("merge_key"),
    F.lit("CLOSE").alias("action"),
    "id", "name", "city",
)
inserts = changed.select(
    (-F.col("id")).alias("merge_key"),
    F.lit("INSERT").alias("action"),
    "id", "name", "city",
)

# New surrogate keys for the inserted versions: max(sk) + row_number()
next_sk = spark.table("scd2_customers").agg({"sk": "max"}).collect()[0][0]
row_num = Window.orderBy("id")
inserts = inserts.withColumn("new_sk", F.lit(next_sk) + F.row_number().over(row_num))

merge_source = closes.unionByName(inserts, allowMissingColumns=True)
merge_source.createOrReplaceTempView("scd2_merge_source")

print("MERGE source (2 rows per changed key: one CLOSE, one INSERT):")
display(merge_source.orderBy("id", "action"))

# =====================================================================
# MERGE — close old versions, insert new ones, atomically
# =====================================================================
spark.sql(f"""
    MERGE INTO scd2_customers AS t
    USING scd2_merge_source AS s
       ON t.id = s.merge_key AND t.is_current = true
    WHEN MATCHED AND s.action = 'CLOSE'
        THEN UPDATE SET t.is_current = false, t.valid_to = '{effective_date}'
    WHEN NOT MATCHED AND s.action = 'INSERT'
        THEN INSERT (sk, id, name, city, valid_from, valid_to, is_current)
             VALUES (s.new_sk, s.id, s.name, s.city, '{effective_date}', NULL, true)
""")

print("TARGET after MERGE (Bob's old version CLOSED, new version appended):")
display(spark.table("scd2_customers").orderBy("id", "valid_from"))

# =====================================================================
# Why the history is useful — point-in-time queries:
# =====================================================================
print("Where did Bob live on 2024-03-15? (point-in-time query)")
display(spark.sql("""
    SELECT id, name, city
    FROM scd2_customers
    WHERE id = 2
      AND valid_from <= '2024-03-15'
      AND (valid_to IS NULL OR valid_to > '2024-03-15')
"""))

print("Current state (what SCD1 would have kept — SCD2 can produce it on demand):")
display(spark.sql("SELECT * FROM scd2_customers WHERE is_current = true ORDER BY id"))

print("""
Production notes:
  * The close+insert pair must land in ONE atomic MERGE, or you end up with
    two current rows for a key (or none) if the job dies mid-flight. Delta
    MERGE guarantees this.
  * valid_to of the old version == valid_from of the new version (no gaps,
    no overlaps) -> use BETWEEN-style point-in-time filters as above.
  * Advance the watermark / process only source rows past the last run, just
    like the incremental examples earlier — unchanged keys cost nothing.
  * For wide tables, compare sha2 row hashes instead of each column (see the
    CDC cell above) before deciding what to close/insert.
  * If your source emits change events (Delta Change Data Feed, CDC streams),
    skip the hand-rolled merge entirely and use APPLY CHANGES INTO with
    TRACK HISTORY ON (SCD Type 2) or track_history_except_columns —
    it manages closing old versions and inserting new ones for you.
""")